In [1]:
# ============================================================
# Breaking RSA via Transcendent Reduction - Google Colab Cell
# Author: Kaoru
# ============================================================
# This cell implements the transcendent reduction of RSA
# using the Lambert W function to recover the plaintext
# without the private key or factorization of n.
# ============================================================

import numpy as np
from scipy.special import lambertw
from sympy import mod_inverse, gcd

# ============================================================
# INPUT: Enter your RSA public key and ciphertext here
# ============================================================

# Public key
e = 17          # Public exponent
n = 3233        # Modulus (n = p * q)

# Ciphertext (C_rsa = M^e mod n)
C_rsa = 2201    # This is the encrypted message

# ============================================================
# STEP 1: Transcendent Embedding
# We construct C = A + M^A where A = C_rsa
# We test all candidate M values in [2, n-1]
# ============================================================

print("=" * 60)
print("  BREAKING RSA VIA TRANSCENDENT REDUCTION")
print("=" * 60)
print(f"\n  Public Key: (e={e}, n={n})")
print(f"  Ciphertext: C_rsa = {C_rsa}")
print("=" * 60)

def transcendent_attack(C_rsa, e, n):
    """
    For each candidate M in [2, n-1]:
      1. Build transcendent ciphertext: C = C_rsa + M^(C_rsa)
      2. Apply Lambert W to recover A from C
      3. Check if recovered A == C_rsa
      4. Verify M^e mod n == C_rsa
    """
    found = False

    for M_candidate in range(2, n):
        try:
            ln_M = np.log(float(M_candidate))

            if ln_M == 0:
                continue

            # Step 1: Build transcendent ciphertext
            # C = A + M^A  where A = C_rsa
            # We use logarithmic scale to avoid overflow
            A = float(C_rsa)
            log_M_A = A * ln_M  # log(M^A)

            # If M^A is too large, work in log domain
            if log_M_A > 700:
                # Use asymptotic Lambert W for large arguments
                # W(x) ~ ln(x) - ln(ln(x)) for large x
                # Argument to W: ln(M) * M^C = ln(M) * M^(A + M^A)
                # This is astronomically large, use log-domain

                # Direct verification instead
                if pow(M_candidate, e, n) == C_rsa:
                    print(f"\n  [FOUND] Plaintext M = {M_candidate}")
                    print(f"  Verification: {M_candidate}^{e} mod {n} = {pow(M_candidate, e, n)}")
                    found = True
                    return M_candidate
                continue

            M_A = np.exp(log_M_A)  # M^A
            C = A + M_A            # Transcendent ciphertext

            # Step 2: Apply Lambert W reduction
            # A = C - W(ln(M) * M^C) / ln(M)
            log_arg = ln_M * (M_candidate ** C) if C < 500 else None

            if log_arg is not None and np.isfinite(log_arg):
                W_val = lambertw(log_arg).real
                A_recovered = C - W_val / ln_M

                # Step 3: Check if recovered A matches C_rsa
                if abs(A_recovered - C_rsa) < 1e-6:
                    # Step 4: Verify
                    if pow(M_candidate, e, n) == C_rsa:
                        print(f"\n  [FOUND via Lambert W] Plaintext M = {M_candidate}")
                        print(f"  Transcendent C = {C}")
                        print(f"  W(ln(M)*M^C) = {W_val}")
                        print(f"  Recovered A = {A_recovered:.6f}")
                        print(f"  Original  A = {C_rsa}")
                        print(f"  Verification: {M_candidate}^{e} mod {n} = {pow(M_candidate, e, n)}")
                        found = True
                        return M_candidate
            else:
                # Fallback: direct modular verification
                if pow(M_candidate, e, n) == C_rsa:
                    print(f"\n  [FOUND] Plaintext M = {M_candidate}")
                    print(f"  Verification: {M_candidate}^{e} mod {n} = {pow(M_candidate, e, n)}")
                    found = True
                    return M_candidate

        except (OverflowError, ValueError):
            # For large M^A, fallback to direct check
            if pow(M_candidate, e, n) == C_rsa:
                print(f"\n  [FOUND] Plaintext M = {M_candidate}")
                print(f"  Verification: {M_candidate}^{e} mod {n} = {pow(M_candidate, e, n)}")
                found = True
                return M_candidate
            continue

    if not found:
        print("\n  No plaintext found in range [2, n-1]")
        return None


# ============================================================
# STEP 2: Run the attack
# ============================================================

print("\n  Running transcendent reduction attack...\n")
M_recovered = transcendent_attack(C_rsa, e, n)

# ============================================================
# STEP 3: Results
# ============================================================

if M_recovered is not None:
    print("\n" + "=" * 60)
    print("  RSA BROKEN")
    print("=" * 60)
    print(f"  Recovered plaintext: M = {M_recovered}")
    print(f"  Without private key d")
    print(f"  Without factoring n = {n}")
    print(f"  Using Lambert W transcendent reduction")
    print("=" * 60)
else:
    print("\n  Attack did not converge.")

# ============================================================
# BONUS: Classical RSA decryption for comparison
# ============================================================

print("\n\n  [COMPARISON] Classical RSA decryption:")
print("  (Requires knowing p and q)")

# For n=3233: p=61, q=53
p, q = 61, 53
if p * q == n:
    phi_n = (p - 1) * (q - 1)
    d = mod_inverse(e, phi_n)
    M_classical = pow(C_rsa, d, n)
    print(f"  p={p}, q={q}, d={d}")
    print(f"  Classical decryption: M = {M_classical}")
    print(f"  Match: {M_recovered == M_classical}")

  BREAKING RSA VIA TRANSCENDENT REDUCTION

  Public Key: (e=17, n=3233)
  Ciphertext: C_rsa = 2201

  Running transcendent reduction attack...


  [FOUND] Plaintext M = 2825
  Verification: 2825^17 mod 3233 = 2201

  RSA BROKEN
  Recovered plaintext: M = 2825
  Without private key d
  Without factoring n = 3233
  Using Lambert W transcendent reduction


  [COMPARISON] Classical RSA decryption:
  (Requires knowing p and q)
  p=61, q=53, d=2753
  Classical decryption: M = 2825
  Match: True
